# Rolling Firm Rate-Sensitivity Targets

This notebook estimates what the embeddings will be asked to recover. At each quarter from 2010Q1 through 2024Q4, each firm's rate sensitivity is estimated using only FOMC events in the preceding five years. No event from the target quarter or the future enters that quarter's label.

## 1. Setup and immutable-input gate

The notebook requires the completed acquisition gate from Notebook 01 and does not connect to WRDS.

In [ ]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
from IPython.display import display

def locate_root():
    start=Path.cwd().resolve()
    for candidate in [start,*start.parents]:
        if (candidate/'full_replication_80q').exists() and (candidate/'economic_shock_sensitivity').exists(): return candidate
    raise FileNotFoundError('Could not locate Asset_Embeddings root.')

ROOT=locate_root(); MODULE=ROOT/'economic_shock_sensitivity'
EVENT_DATA=MODULE/'data'/'processed'/'fomc_event_returns'; TARGET_ROOT=MODULE/'data'/'processed'/'rolling_rate_targets'
RESULTS=MODULE/'results'/'rolling_rate_targets'; MANIFESTS=MODULE/'manifests'
for path in [TARGET_ROOT,RESULTS,MANIFESTS]: path.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(MODULE/'src'))
import rate_sensitivity_targets as rst
from rate_shock_audit import permanent_split
gate=pd.read_parquet(MODULE/'results'/'event_return_acquisition'/'acquisition_gate.parquet')
display(gate); assert gate.passed.all(), 'Notebook 01 acquisition gate has not passed.'
firm_events=pd.read_parquet(EVENT_DATA/'firm_fomc_event_returns_2005_2024.parquet')
firm_events['event_date']=pd.to_datetime(firm_events.event_date)
print(f'Loaded {len(firm_events):,} firm-event observations across {firm_events.event_date.nunique()} events.')

## 2. Freeze target specifications

The confirmatory target regresses announcement-day market-adjusted firm returns on the statement-window surprise. Two robustness targets change one element at a time: the return window or the shock window. The surprise is kept in percentage points, so beta is the return response to a one-percentage-point surprise; typical shocks are much smaller.

In [ ]:
TARGET_CONTRACT={
 'version':'rolling-rate-target-v1', 'anchors':['2010Q1','2024Q4'], 'lookback_years':5, 'minimum_events':20,
 'event_inclusion':'event_date >= anchor_start minus 5 years and event_date < anchor_start',
 'primary':{'name':'stmt_ar00','shock':'shock_stmt','outcome':'abnormal_return_00'},
 'robustness':[{'name':'stmt_ar01','shock':'shock_stmt','outcome':'abnormal_return_01'},{'name':'me_ar00','shock':'shock_me','outcome':'abnormal_return_00'}],
 'regression':'firm-level OLS with intercept across historical events',
 'winsorization':'cross-sectional 1st/99th percentiles within anchor and specification',
 'shrinkage':'empirical-Bayes reliability ratio using estimated cross-sectional signal variance and firm OLS standard error',
 'primary_label_for_prediction':'beta_shrunk', 'raw_label_retained':True, 'permanent_firm_split_seed':'rate-shock-v1'}
(MANIFESTS/'rolling_rate_target_contract_v1.json').write_text(json.dumps(TARGET_CONTRACT,indent=2,sort_keys=True)+'\n')
display(pd.json_normalize(TARGET_CONTRACT,sep='.').T.rename(columns={0:'value'}))

## 3. Estimation function

For a given anchor, the function first restricts the firm universe to firms present in the previous-quarter RS-Binary embedding. It then computes OLS coefficients, standard errors, and R² using sufficient statistics. Finally it winsorizes the cross-section and shrinks noisy estimates toward the quarterly mean.

In [ ]:
specifications=[TARGET_CONTRACT['primary'],*TARGET_CONTRACT['robustness']]
anchors=pd.period_range('2010Q1','2024Q4',freq='Q')

def estimate_anchor(anchor,specification):
    label=str(anchor).lower(); embedding_label=str(anchor-1).lower(); anchor_start=anchor.start_time.normalize(); history_start=anchor_start-pd.DateOffset(years=5)
    embedding_path=ROOT/'full_replication_80q'/'embeddings'/embedding_label/'rs_binary'/'asset_embeddings_normalized_labeled.parquet'
    universe=pd.read_parquet(embedding_path,columns=['permco','firm_name']).drop_duplicates('permco')
    history=firm_events.loc[firm_events.event_date.ge(history_start)&firm_events.event_date.lt(anchor_start)&firm_events.permco.isin(universe.permco)].copy()
    estimates=rst.estimate_grouped_betas(history,specification['shock'],specification['outcome'],TARGET_CONTRACT['minimum_events'])
    estimates=universe.merge(estimates,on='permco',how='left',validate='one_to_one'); estimates['eligible']=estimates.eligible.fillna(False)
    estimates,audit=rst.add_cross_sectional_shrinkage(estimates)
    estimates['anchor_quarter']=label; estimates['embedding_quarter']=embedding_label; estimates['history_start']=history_start; estimates['history_end_exclusive']=anchor_start
    estimates['specification']=specification['name']; estimates['firm_split']=permanent_split(estimates.permco)
    audit.update(anchor_quarter=label,embedding_quarter=embedding_label,specification=specification['name'],universe_firms=len(universe),historical_events=history.event_date.nunique(),history_start=history_start,history_end_exclusive=anchor_start)
    columns=['anchor_quarter','embedding_quarter','history_start','history_end_exclusive','specification','permco','firm_name','firm_split','events','beta_raw','intercept','beta_standard_error','regression_r2','eligible','beta_winsorized','reliability','beta_shrunk']
    return estimates[columns],audit


## 4. Estimate and checkpoint all 60 quarters

Each quarter/specification is an independent checkpoint. Rerunning this cell skips completed files. This step is local and should ordinarily take only a few minutes.

In [ ]:
audit_rows=[]; target_frames=[]
for number,anchor in enumerate(anchors,1):
    for specification in specifications:
        label=str(anchor).lower(); directory=TARGET_ROOT/label; directory.mkdir(parents=True,exist_ok=True); path=directory/f"{specification['name']}.parquet"
        started=time.perf_counter()
        if path.exists():
            estimates=pd.read_parquet(path); status='checkpoint'
            eligible=estimates[estimates.eligible]
            audit={'anchor_quarter':label,'embedding_quarter':str(anchor-1).lower(),'specification':specification['name'],'universe_firms':len(estimates),'eligible_firms':len(eligible),'historical_events':int(eligible.events.max()) if len(eligible) else 0,'median_reliability':eligible.reliability.median(),'history_start':anchor.start_time-pd.DateOffset(years=5),'history_end_exclusive':anchor.start_time}
        else:
            estimates,audit=estimate_anchor(anchor,specification); estimates.to_parquet(path,index=False); status='estimated'
        audit.update(status=status,seconds=time.perf_counter()-started); audit_rows.append(audit); target_frames.append(estimates)
    print(f'{number:02d}/60 {label}: complete')
target_audit=pd.DataFrame(audit_rows); all_targets=pd.concat(target_frames,ignore_index=True)
target_audit.to_parquet(RESULTS/'target_estimation_audit.parquet',index=False); all_targets.to_parquet(TARGET_ROOT/'all_rolling_rate_targets.parquet',index=False)
display(target_audit)

## 5. Coverage and estimation precision

Coverage is reported relative to the actual embedding universe. Reliability near one indicates precise firm estimates; values near zero indicate that sampling noise dominates the observed cross-sectional dispersion. Both raw and shrunk betas remain available so later results cannot depend on an undocumented transformation.

In [ ]:
coverage=(all_targets.groupby(['anchor_quarter','specification'],as_index=False).agg(universe_firms=('permco','size'),eligible_firms=('eligible','sum'),median_events=('events','median'),median_standard_error=('beta_standard_error','median'),median_r2=('regression_r2','median'),median_reliability=('reliability','median')))
coverage['coverage_rate']=coverage.eligible_firms/coverage.universe_firms
coverage.to_parquet(RESULTS/'target_coverage_by_quarter.parquet',index=False)
display(coverage.groupby('specification',as_index=False).agg(quarters=('anchor_quarter','nunique'),minimum_coverage=('coverage_rate','min'),mean_coverage=('coverage_rate','mean'),median_events=('median_events','median'),median_standard_error=('median_standard_error','median'),median_r2=('median_r2','median'),median_reliability=('median_reliability','median')))
display(coverage.tail(12))

## 6. Target distributions

The following summaries make scale and shrinkage visible. A positive beta means the firm tends to outperform the market when policy is unexpectedly tighter; a negative beta means it tends to underperform.

In [ ]:
eligible_targets=all_targets[all_targets.eligible].copy()
distribution=(eligible_targets.groupby('specification')[['beta_raw','beta_winsorized','beta_shrunk','beta_standard_error','regression_r2','reliability']].describe(percentiles=[.01,.1,.5,.9,.99]))
distribution.to_parquet(RESULTS/'target_distributions.parquet')
display(distribution)
latest=eligible_targets.query("anchor_quarter=='2024q4' and specification=='stmt_ar00'")
display(latest.nlargest(10,'beta_shrunk')[['permco','firm_name','events','beta_raw','beta_shrunk','reliability']])
display(latest.nsmallest(10,'beta_shrunk')[['permco','firm_name','events','beta_raw','beta_shrunk','reliability']])

## 7. Adjacent-quarter stability

This is a measurement diagnostic, not an independent predictive test: adjacent five-year windows share most events mechanically. Extremely low stability would nevertheless reveal an implementation or measurement problem.

In [ ]:
stability=[]
for specification,group in eligible_targets.groupby('specification'):
    labels=list(map(str,anchors.astype(str))); labels=[x.lower() for x in labels]
    for previous,current in zip(labels[:-1],labels[1:]):
        left=group[group.anchor_quarter.eq(previous)][['permco','beta_raw','beta_shrunk']]; right=group[group.anchor_quarter.eq(current)][['permco','beta_raw','beta_shrunk']]
        joined=left.merge(right,on='permco',suffixes=('_previous','_current'))
        stability.append({'specification':specification,'previous_quarter':previous,'current_quarter':current,'common_firms':len(joined),'raw_pearson':joined.beta_raw_previous.corr(joined.beta_raw_current),'shrunk_pearson':joined.beta_shrunk_previous.corr(joined.beta_shrunk_current),'shrunk_spearman':joined.beta_shrunk_previous.corr(joined.beta_shrunk_current,method='spearman')})
stability=pd.DataFrame(stability); stability.to_parquet(RESULTS/'adjacent_quarter_stability.parquet',index=False)
display(stability.groupby('specification')[['common_firms','raw_pearson','shrunk_pearson','shrunk_spearman']].describe(percentiles=[.1,.5,.9]))

## 8. Target-construction gate

Passing this gate means the labels are available and sufficiently populated. It does not establish that embeddings predict them; that question remains untouched until the next notebook.

In [ ]:
primary_coverage=coverage[coverage.specification.eq('stmt_ar00')]
primary_targets=eligible_targets[eligible_targets.specification.eq('stmt_ar00')]
target_gate=pd.DataFrame([
 {'test':'60 primary anchor quarters','value':primary_coverage.anchor_quarter.nunique(),'passed':primary_coverage.anchor_quarter.nunique()==60},
 {'test':'minimum primary coverage >= 90%','value':primary_coverage.coverage_rate.min(),'passed':primary_coverage.coverage_rate.min()>=.90},
 {'test':'all eligible estimates use >=20 events','value':primary_targets.events.min(),'passed':primary_targets.events.min()>=20},
 {'test':'finite primary shrunk targets','value':int(np.isfinite(primary_targets.beta_shrunk).sum()),'passed':np.isfinite(primary_targets.beta_shrunk).all()},
 {'test':'median adjacent-quarter rank stability >= 0.50','value':stability.loc[stability.specification.eq('stmt_ar00'),'shrunk_spearman'].median(),'passed':stability.loc[stability.specification.eq('stmt_ar00'),'shrunk_spearman'].median()>=.50},
])
target_gate.to_parquet(RESULTS/'target_construction_gate.parquet',index=False); display(target_gate)
print('Combined targets:',TARGET_ROOT/'all_rolling_rate_targets.parquet')
print('STOP: inspect target quality before testing any embedding model.')